# Parking Birmingham — Data Audit and Validation

**Course:** ADSP 32021 — Machine Learning Operations  
**Project:** Parking Availability Prediction Using an Automated MLOps Pipeline

## 1. Introduction

This notebook performs an initial audit of the UCI Parking Birmingham dataset to assess data quality, structure, and suitability for forecasting parking occupancy.

The audit will examine:
- Dataset dimensions, column names, and data types
- Missing values and duplicate observations
- Timestamp coverage and observation frequency
- Parking capacity and occupancy constraints
- Feasibility of a 30-minute prediction horizon

The findings will inform preprocessing, feature engineering, and chronological data splitting.

**Data source:** [UCI Parking Birmingham](https://archive.ics.uci.edu/dataset/482/parking+birmingham)


## 2. Dataset Loading and Initial Inspection

We begin by loading the raw dataset and examining its dimensions, column names, data types, and sample observations.

This initial inspection helps identify potential formatting issues and establishes a baseline understanding of the available variables before applying any transformations.

In [1]:
import pandas as pd
from pathlib import Path

# Path to the raw dataset
data_path = Path("../data/dataset.csv")

# Load the dataset
df = pd.read_csv(data_path)

# Inspect dataset structure
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

# Display the first five observations
display(df.head())


Dataset shape: (35717, 4)

Column names: ['SystemCodeNumber', 'Capacity', 'Occupancy', 'LastUpdated']

Data types:
SystemCodeNumber    object
Capacity             int64
Occupancy            int64
LastUpdated         object
dtype: object


,SystemCodeNumber,Capacity,Occupancy,LastUpdated
0,BHMBCCMKT01,577,61,2016-10-04 07:59:42
1,BHMBCCMKT01,577,64,2016-10-04 08:25:42
2,BHMBCCMKT01,577,80,2016-10-04 08:59:42
3,BHMBCCMKT01,577,107,2016-10-04 09:32:46
4,BHMBCCMKT01,577,150,2016-10-04 09:59:48


### 2.1 Initial Inspection Findings

The dataset contains **35,717 observations and 4 variables**:

- **SystemCodeNumber (object):** Unique identifier for each parking facility.
- **Capacity (int64):** Total number of parking spaces at a facility.
- **Occupancy (int64):** Number of occupied parking spaces at the recorded time.
- **LastUpdated (object):** Timestamp indicating when the observation was recorded.

**Key observations:**

1. The dataset contains the expected number of observations and variables.
2. Capacity and Occupancy are already stored as integers, making them suitable for numerical analysis.
3. LastUpdated is currently stored as an object rather than a datetime type and will require conversion.
4. SystemCodeNumber is a categorical identifier and should not be treated as a continuous numerical variable.

**Next steps:** Evaluate missing values, duplicate records, timestamp consistency, and occupancy constraints before preprocessing.

## 3. Missing Values and Duplicate Records

Missing values can affect model training and feature engineering, particularly when constructing time-dependent features.

Duplicate observations may distort historical parking patterns or create misleading training examples.

We examine missing values across all variables, count exact duplicate rows, and check for repeated facility-timestamp combinations.


In [2]:
# Check missing values
print("Missing values by column:")
print(df.isnull().sum())

print("\nMissing value percentages:")
print((df.isnull().mean() * 100).round(2))

# Check duplicate records
print("\nExact duplicate rows:", df.duplicated().sum())

# Check duplicate facility-timestamp combinations
duplicate_keys = df.duplicated(
    subset=["SystemCodeNumber", "LastUpdated"]
).sum()

print("Duplicate facility-timestamp combinations:", duplicate_keys)

# Count unique parking facilities
print("\nUnique parking facilities:", df["SystemCodeNumber"].nunique())


Missing values by column:
SystemCodeNumber    0
Capacity            0
Occupancy           0
LastUpdated         0
dtype: int64

Missing value percentages:
SystemCodeNumber    0.0
Capacity            0.0
Occupancy           0.0
LastUpdated         0.0
dtype: float64

Exact duplicate rows: 216
Duplicate facility-timestamp combinations: 216

Unique parking facilities: 30


### 3.1 Missing Values and Duplicate Records — Findings

The dataset contains **35,717 observations across 30 parking facilities**.

**Missing values:**
- No missing values were identified in any of the four variables.
- No missing-value imputation is required for the original dataset.

**Duplicate records:**
- 216 exact duplicate rows were identified.
- 216 duplicate facility-timestamp combinations were identified.
- These duplicates may distort observed parking patterns and introduce redundant training examples.

**Cleaning decision:**

Exact duplicate records will be removed during preprocessing, retaining one observation per unique record. The expected cleaned dataset contains 35,501 observations.

The original CSV will remain unchanged to preserve reproducibility. Duplicate removal will be implemented in the preprocessing pipeline so the same transformation can be applied automatically during future pipeline executions.


## 4. Initial Data Cleaning

Exact duplicate observations are removed to prevent redundant records from affecting model development.

The raw dataset is preserved, and cleaning is performed on a separate DataFrame to maintain reproducibility and traceability.

Additional validation, including timestamp consistency and occupancy constraints, will be performed before the dataset is considered ready for modeling.


In [3]:
# Create a cleaned copy of the dataset
df_clean = df.drop_duplicates().copy()

# Verify duplicate removal
print("Original observations:", len(df))
print("Cleaned observations:", len(df_clean))
print("Removed duplicates:", len(df) - len(df_clean))

print("\nRemaining exact duplicates:",
      df_clean.duplicated().sum())

print("Remaining duplicate facility-timestamp combinations:",
      df_clean.duplicated(
          subset=["SystemCodeNumber", "LastUpdated"]
      ).sum())


Original observations: 35717
Cleaned observations: 35501
Removed duplicates: 216

Remaining exact duplicates: 0
Remaining duplicate facility-timestamp combinations: 0


### 4.1 Duplicate Removal Results

After removing exact duplicates, the dataset contains **35,501 observations**.

- **Original observations:** 35,717
- **Duplicate records removed:** 216
- **Remaining exact duplicates:** 0
- **Remaining duplicate facility-timestamp combinations:** 0

The cleaned dataset will be used for subsequent validation and exploratory analysis. The original dataset remains unchanged.

## 5. Timestamp Validation

Accurate timestamps are essential for time-series forecasting because observations must be ordered chronologically and historical features must not contain future information.

This section examines:
- Timestamp conversion and parsing errors
- Earliest and latest observation dates
- Observation intervals within each parking facility
- Whether the data supports a 30-minute forecasting horizon

Timestamp validation will guide the selection of an appropriate prediction horizon and the construction of leakage-safe lag features.


In [4]:
# Convert timestamps to datetime
df_clean["LastUpdated"] = pd.to_datetime(
    df_clean["LastUpdated"],
    errors="coerce"
)

# Check for invalid timestamps
print("Invalid timestamps:",
      df_clean["LastUpdated"].isna().sum())

# Check dataset date range
print("\nEarliest timestamp:",
      df_clean["LastUpdated"].min())

print("Latest timestamp:",
      df_clean["LastUpdated"].max())

# Sort observations chronologically within each facility
df_clean = df_clean.sort_values(
    ["SystemCodeNumber", "LastUpdated"]
).reset_index(drop=True)

# Calculate time between consecutive observations per facility
df_clean["TimeDifferenceMinutes"] = (
    df_clean.groupby("SystemCodeNumber")["LastUpdated"]
    .diff()
    .dt.total_seconds() / 60
)

# Summarize observation intervals
print("\nObservation interval statistics (minutes):")
print(df_clean["TimeDifferenceMinutes"].describe())

# Most common observation intervals
print("\nMost common observation intervals (rounded minutes):")
print(
    df_clean["TimeDifferenceMinutes"]
    .round()
    .value_counts()
    .head(10)
)


Invalid timestamps: 0

Earliest timestamp: 2016-10-04 07:46:28
Latest timestamp: 2016-12-19 16:30:35

Observation interval statistics (minutes):
count    35471.000000
mean        88.607939
std        322.241946
min          5.966667
25%         26.983333
50%         32.933333
75%         33.066667
max      13908.783333
Name: TimeDifferenceMinutes, dtype: float64

Most common observation intervals (rounded minutes):
TimeDifferenceMinutes
27.0     10148
33.0     10044
26.0      5276
34.0      5226
30.0       601
930.0      456
20.0       429
40.0       414
933.0      308
31.0       256
Name: count, dtype: int64


### 5.1 Timestamp Validation Findings

All 35,501 cleaned observations have valid timestamps. The dataset covers October 4 through December 19, 2016.

**Observation frequency:**
- Median interval: 32.93 minutes
- Mean interval: 88.61 minutes
- Minimum interval: 5.97 minutes
- Maximum interval: approximately 9.66 days

The most common observation intervals are approximately 26–27 and 33–34 minutes. This indicates that observations are irregularly spaced rather than recorded at exact 30-minute intervals.

Substantial gaps of approximately 930 minutes are also present. These may represent overnight periods without observations, although their cause has not yet been confirmed.

**Implications for forecasting:**

A simple one-row shift would not always represent a 30-minute forecast because consecutive observations may be separated by substantially different time intervals.

We will investigate observation gaps and determine an appropriate timestamp-aware method for constructing future occupancy targets.

The proposed 30-minute prediction horizon remains provisional until its feasibility has been verified.


### 5.2 Observation Gap Analysis

Irregular observation intervals can introduce errors when constructing future occupancy targets.

We examine the frequency of large gaps within each parking facility to determine whether missing reporting periods are isolated events or recurring patterns.

This analysis will help establish acceptable timestamp tolerances for future target construction and prevent observations separated by several hours or days from being treated as consecutive 30-minute measurements.


In [5]:
# Categorize intervals between consecutive observations
intervals = df_clean["TimeDifferenceMinutes"].dropna()

gap_summary = pd.Series({
    "Intervals <= 40 minutes": (intervals <= 40).sum(),
    "Intervals > 40 to 60 minutes": (
        (intervals > 40) & (intervals <= 60)
    ).sum(),
    "Intervals > 60 to 180 minutes": (
        (intervals > 60) & (intervals <= 180)
    ).sum(),
    "Intervals > 180 minutes": (intervals > 180).sum()
})

gap_summary_df = pd.DataFrame({
    "Count": gap_summary,
    "Percentage": (gap_summary / len(intervals) * 100).round(2)
})

display(gap_summary_df)

# Examine the largest observation gaps
print("\nTen largest gaps:")
display(
    df_clean.nlargest(
        10, "TimeDifferenceMinutes"
    )[
        ["SystemCodeNumber", "LastUpdated",
         "TimeDifferenceMinutes"]
    ]
)


,Count,Percentage
Intervals <= 40 minutes,33063,93.21
Intervals > 40 to 60 minutes,390,1.10
Intervals > 60 to 180 minutes,60,0.17
Intervals > 180 minutes,1958,5.52



Ten largest gaps:


,SystemCodeNumber,LastUpdated,TimeDifferenceMinutes
23741,NIA North,2016-10-28 08:16:41,13908.783333
23828,NIA North,2016-11-20 08:01:19,13894.100000
23846,NIA North,2016-11-30 08:02:35,13891.233333
23792,NIA North,2016-11-07 08:00:52,11067.883333
5191,BHMBRCBRG01,2016-10-12 08:04:00,11012.766667
6370,BHMBRCBRG02,2016-10-12 08:04:00,11012.766667
7550,BHMBRCBRG03,2016-10-12 08:04:00,11012.766667
21343,Bull Ring,2016-10-12 08:04:00,11012.766667
15481,BHMNCPNHS01,2016-10-24 07:56:56,6691.883333
23617,NIA Car Parks,2016-12-12 08:02:45,5250.300000


### 5.3 Observation Gap Analysis Findings

Of the 35,471 intervals between consecutive observations within parking facilities:

- **93.21%** are 40 minutes or shorter.
- **1.10%** are between 40 and 60 minutes.
- **0.17%** are between 60 and 180 minutes.
- **5.52%** exceed 180 minutes.

The largest observed gap is approximately 9.66 days, with several substantial gaps occurring at the NIA North facility.

**Implications:**

Most observations occur at intervals broadly consistent with approximately half-hourly reporting. However, the presence of overnight and multi-day gaps makes a simple unconditional row shift inappropriate for constructing forecasting targets.

We will evaluate the number of observations with a subsequent measurement within a defined tolerance of 30 minutes before finalizing the forecasting horizon.

Large gaps will not be bridged through unrestricted interpolation or forward filling.


### 5.4 Feasibility of a 30-Minute Forecasting Horizon

The proposed forecasting objective is to predict parking occupancy approximately 30 minutes into the future.

Because observations are not recorded at perfectly regular intervals, we evaluate how frequently the next observation for a facility occurs within a tolerance of 25–35 minutes.

This analysis estimates the number of potential supervised learning examples and helps determine whether the proposed horizon is practical without introducing targets across long reporting gaps.

In [6]:
# Calculate the interval to the next observation for each facility
df_clean["NextTimestamp"] = (
    df_clean.groupby("SystemCodeNumber")["LastUpdated"].shift(-1)
)

df_clean["MinutesToNext"] = (
    (df_clean["NextTimestamp"] - df_clean["LastUpdated"])
    .dt.total_seconds() / 60
)

# Check how many next observations fall within 25–35 minutes
valid_horizon = df_clean["MinutesToNext"].between(25, 35)

print("Total cleaned observations:", len(df_clean))
print("Observations with a next timestamp:",
      df_clean["MinutesToNext"].notna().sum())

print("Potential 30-minute forecast examples:",
      valid_horizon.sum())

print("Percentage of cleaned observations:",
      round(valid_horizon.mean() * 100, 2), "%")

# Check coverage by parking facility
facility_coverage = (
    df_clean.assign(ValidHorizon=valid_horizon)
    .groupby("SystemCodeNumber")["ValidHorizon"]
    .agg(["sum", "count", "mean"])
)

facility_coverage["mean"] = (
    facility_coverage["mean"] * 100
).round(2)

facility_coverage = facility_coverage.rename(
    columns={
        "sum": "ValidExamples",
        "count": "TotalObservations",
        "mean": "CoveragePercent"
    }
)

display(facility_coverage.sort_values("CoveragePercent"))


Total cleaned observations: 35501
Observations with a next timestamp: 35471
Potential 30-minute forecast examples: 31801
Percentage of cleaned observations: 89.58 %


,ValidExamples,TotalObservations,CoveragePercent
SystemCodeNumber,,,
BHMNCPPLS01,994,1253,79.33
BHMNCPRAN01,979,1174,83.39
BHMNCPNHS01,870,1028,84.63
BHMNCPNST01,1101,1299,84.76
BHMNCPLDH01,1106,1281,86.34
BHMNCPHST01,1149,1301,88.32
BHMBRTARC01,79,88,89.77
BHMBRCBRG02,1061,1180,89.92
NIA North,143,159,89.94
